<a href="https://colab.research.google.com/github/hetaf234/GP_Layers/blob/main/Terra_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## DataSet

### Food101

In [36]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dansbecker/food-101")

print("Path to dataset files:", path)


KeyboardInterrupt: 

In [ ]:
images_path = os.path.join(path, "images")

print(images_path)

### Recipes 5K

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("terry9a9/recipes5k")

print("Path to dataset files:", path)


In [ ]:
with open("/kaggle/input/recipes5k/Recipes5k/annotations/ingredients_Recipes5k.txt") as f:
    for _ in range(10):
        print(f.readline())

In [ ]:
recipes.columns

## Terra Model

In [38]:
!pip install -U openai kagglehub pillow

In [39]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

Connection Test

In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

response = client.responses.create(
    model="gpt-5.5",
    input="Say hello"
)

print(response.output_text)

## Library uploading

In [37]:
import os
import json
import pandas as pd
import kagglehub
import base64
import mimetypes

from openai import OpenAI
from google.colab import userdata
from PIL import Image
from google.colab import files



## Upload image

In [40]:
from google.colab import files
uploaded = files.upload()

image_path = next(iter(uploaded.keys()))

Image.open(image_path)
mime_type = mimetypes.guess_type(image_path)[0] or "image/jpeg"

with open(image_path, "rb") as f:
    image_base64 = base64.b64encode(f.read()).decode("utf-8")

Saving images (1).jpg to images (1) (1).jpg


## Prompt - Send Image to GPT

In [41]:
schema = {
    "name": "food_analysis",
    "schema": {
        "type": "object",
        "properties": {
            "dish": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "confidence": {"type": "number"}
                },
                "required": ["name", "confidence"],
                "additionalProperties": False
            },
            "visible_ingredients": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string"},
                        "confidence": {"type": "number"}
                    },
                    "required": ["name", "confidence"],
                    "additionalProperties": False
                }
            },
            "predicted_allergens": {
                "type": "object",
                "properties": {
                    "dairy": {"type": "boolean"},
                    "eggs": {"type": "boolean"},
                    "fish": {"type": "boolean"},
                    "crustacean_shellfish": {"type": "boolean"},
                    "tree_nuts": {"type": "boolean"},
                    "peanuts": {"type": "boolean"},
                    "wheat_gluten": {"type": "boolean"},
                    "soy": {"type": "boolean"},
                    "sesame": {"type": "boolean"},
                    "mustard": {"type": "boolean"},
                    "celery": {"type": "boolean"},
                    "sulphites": {"type": "boolean"}
                },
                "required": [
                    "dairy",
                    "eggs",
                    "fish",
                    "crustacean_shellfish",
                    "tree_nuts",
                    "peanuts",
                    "wheat_gluten",
                    "soy",
                    "sesame",
                    "mustard",
                    "celery",
                    "sulphites"
                ],
                "additionalProperties": False
            },
            "notes": {
                "type": "string"
            }
        },
        "required": [
            "dish",
            "visible_ingredients",
            "predicted_allergens",
            "notes"
        ],
        "additionalProperties": False
    }
}

In [45]:
prompt = """
You are an expert food recognition and food allergy analysis AI.

Analyze the uploaded food image carefully.

Your tasks are:

1. Identify the dish name.
2. Return your confidence score (0-1).
3. Detect the visible ingredients only.
4. Return a confidence score for each visible ingredient.
5. Infer common hidden ingredients that are typically used in this dish, even if they are not visually observable.
6. Predict allergens based on BOTH:
   - visible ingredients
   - commonly used hidden ingredients in this type of dish.
7. If an allergen cannot be visually confirmed but is commonly present in this dish, still mark it as true.
8. Add a short explanation in the notes explaining which allergens are inferred rather than visually confirmed.

Return ONLY valid JSON using exactly this structure:

{
  "dish": {
    "name": "",
    "confidence": 0.0
  },
  "visible_ingredients": [
    {
      "name": "",
      "confidence": 0.0
    }
  ],
  "predicted_allergens": {
    "dairy": false,
    "eggs": false,
    "fish": false,
    "crustacean_shellfish": false,
    "tree_nuts": false,
    "peanuts": false,
    "wheat_gluten": false,
    "soy": false,
    "sesame": false,
    "mustard": false,
    "celery": false,
    "sulphites": false
  },
  "notes": ""
}

Important rules:

- If a dish commonly contains eggs, gluten, dairy, soy, sesame or other allergens, mark them as true even if they cannot be directly seen.
- Prefer food safety over omission. Missing a likely allergen is worse than flagging a possible one.
- Use culinary knowledge to infer ingredients commonly used in standard recipes.
- Do not invent extremely uncommon ingredients.
- Return JSON only.
"""

In [46]:
response = client.responses.create(
    model="gpt-5.5",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": prompt
                },
                {
                    "type": "input_image",
                    "image_url": f"data:{mime_type};base64,{image_base64}"
                }
            ]
        }
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "food_analysis",
            "schema": schema["schema"],
            "strict": True
        }
    }
)

In [47]:
result = json.loads(response.output_text)

print(json.dumps(result, indent=4))

{
    "dish": {
        "name": "Pepperoni and vegetable pizza",
        "confidence": 0.93
    },
    "visible_ingredients": [
        {
            "name": "pizza crust",
            "confidence": 0.95
        },
        {
            "name": "melted cheese",
            "confidence": 0.95
        },
        {
            "name": "tomato sauce",
            "confidence": 0.88
        },
        {
            "name": "pepperoni",
            "confidence": 0.92
        },
        {
            "name": "green bell pepper",
            "confidence": 0.84
        },
        {
            "name": "black olives",
            "confidence": 0.82
        },
        {
            "name": "mushrooms",
            "confidence": 0.76
        },
        {
            "name": "herbs or seasoning",
            "confidence": 0.7
        }
    ],
    "predicted_allergens": {
        "dairy": true,
        "eggs": false,
        "fish": false,
        "crustacean_shellfish": false,
        "tree_nuts": 

## Allergy Detection

In [ ]:
allergy_db = {
    "Milk":["Cheese","Butter","Cream","Milk","Yogurt"],
    "Gluten":["Bread","Flour","Pasta","Wheat"],
    "Egg":["Egg","Mayonnaise"],
    "Soy":["Soy Sauce","Soybean"],
    "Peanut":["Peanut"],
    "Tree Nuts":["Almond","Walnut","Cashew"],
    "Fish":["Fish"],
    "Shellfish":["Shrimp","Crab","Lobster"],
    "Sesame":["Sesame"],
    "Mustard":["Mustard"],
    "Celery":["Celery"],
    "Sulphites":["Wine","Vinegar"]
}

## Result

In [ ]:
final_output = {
    "dish": result["dish"],
    "visible_ingredients": result["visible_ingredients"],

    "detected_allergens": detected
}

print(json.dumps(final_output, indent=4))

In [ ]:
with open("food_result.json","w") as f:
    json.dump(final_output,f,indent=4)

print("Saved!")